<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Refreshing pages actually works

The FlyRank report reports that 7 of 9 tested strata showed statistically significant refresh lift. It also reports a median effect of +588 impressions for refreshed versus stale pages among pages older than 180 days.

My methodology question is: how exactly is the refreshed-versus-stale label defined, and are the refresh status and the outcome measurement separated in time? I would want the refresh group to be defined using information available before the outcome window begins. I would also check whether differences between refreshed and stale pages could explain part of the measured lift.

The result is useful as an observed association in the reported portfolio data. A stronger causal interpretation would require a design that more fully controls for differences between the groups.


### Finding 2 — Dead-page recovery

The FlyRank report states that 65.7K pages had zero traffic in the previous month and that 59% later recovered. It also reports a recovery model with 99% accuracy on unseen pages from the same brands and 97% on unseen brands.

My methodology question is: how is the recovery label defined, and are all model features available before the recovery period starts? I would check that no information from the recovery period is used to construct the predictors or define the population.

The same-brand and unseen-brand evaluations provide useful checks of generalization. I would still want the exact prediction window, feature window, and label construction to be explicit before treating the result as forward-looking decision support.

In [1]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

paper_audit = pd.DataFrame({
    "Finding": [
        "Refreshing pages",
        "Dead-page recovery"
    ],
    "Reported evidence": [
        "7 of 9 strata significant; +588 median effect for 180+ day pages",
        "59% recovery; 99% same-brand and 97% unseen-brand model accuracy"
    ],
    "Methodology question": [
        "Is refresh status defined before the outcome window?",
        "Are all predictors available before the recovery window?"
    ]
})

display(paper_audit)

print("Paper-audit checks recorded:", len(paper_audit))
print("Each finding has a methodology question:",
      paper_audit["Methodology question"].notna().all())

,Finding,Reported evidence,Methodology question
0,Refreshing pages,7 of 9 strata significant; +588 median effect ...,Is refresh status defined before the outcome w...
1,Dead-page recovery,59% recovery; 99% same-brand and 97% unseen-br...,Are all predictors available before the recove...


Paper-audit checks recorded: 2
Each finding has a methodology question: True


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation design

My Week-5 model used a random stratified 80/20 row split. That split can place pages from the same client in both training and testing.

For this audit, I use a client-grouped 80/20 split. Each client is assigned entirely to either the training set or the test set. This tests whether the model generalizes to clients that were not represented during training.

I keep the same target, the same 19 features, the same Logistic Regression model, and the same evaluation metrics. The main change is the validation design.

The Week-5 random-split result is the "before" measurement. The client-grouped result is the "after" measurement.

The comparison measures sensitivity to validation design. It should not be interpreted as proof of future production performance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 2 — HONEST CLIENT-GROUPED VALIDATION
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. Load the same dataset used in Week 5
# ------------------------------------------------------------

repo = "/content/flyrank-ml-internship-starter"

if not os.path.exists(repo):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

raw_path = os.path.join(
    repo,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(raw_path)

print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 2. Create the same Week-5 target
# ------------------------------------------------------------

df["target"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)


# ------------------------------------------------------------
# 3. Use the SAME 19 Week-5 features
# ------------------------------------------------------------

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

print("Number of features:", len(feature_cols))


# ------------------------------------------------------------
# 4. Prepare numeric features
# ------------------------------------------------------------

feature_df = df[
    feature_cols + ["target", "client_id"]
].copy()

for col in feature_cols:
    feature_df[col] = pd.to_numeric(
        feature_df[col],
        errors="coerce"
    ).fillna(0)

X = feature_df[feature_cols]
y = feature_df["target"]
groups = feature_df["client_id"]


# ------------------------------------------------------------
# 5. BEFORE — reproduce Week-5 random split
# ------------------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

before_roc_auc = roc_auc_score(
    y_test_random,
    random_scores
)

before_ap = average_precision_score(
    y_test_random,
    random_scores
)


# ------------------------------------------------------------
# 6. AFTER — client-grouped split
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


# ------------------------------------------------------------
# 7. Check that clients do not overlap
# ------------------------------------------------------------

train_clients = set(groups_train)
test_clients = set(groups_test)

client_overlap = train_clients.intersection(
    test_clients
)

print("\nCLIENT GROUP CHECK")
print("------------------")
print("Training rows:", len(X_train_group))
print("Testing rows:", len(X_test_group))
print("Training clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())
print("Client overlap:", len(client_overlap))


# ------------------------------------------------------------
# 8. Train the same Logistic Regression model
# ------------------------------------------------------------

group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

group_model.fit(
    X_train_group,
    y_train_group
)

group_scores = group_model.predict_proba(
    X_test_group
)[:, 1]


# ------------------------------------------------------------
# 9. AFTER metrics
# ------------------------------------------------------------

after_roc_auc = roc_auc_score(
    y_test_group,
    group_scores
)

after_ap = average_precision_score(
    y_test_group,
    group_scores
)


# ------------------------------------------------------------
# 10. BEFORE vs AFTER table
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Evaluation": [
        "Week-5 random split",
        "ML-09 client-grouped split"
    ],
    "ROC-AUC": [
        before_roc_auc,
        after_roc_auc
    ],
    "Average Precision": [
        before_ap,
        after_ap
    ]
})

print("\nBEFORE vs AFTER")
print("----------------")

display(
    comparison.round(4)
)


# ------------------------------------------------------------
# 11. Difference
# ------------------------------------------------------------

print("Metric change under grouped validation:")
print(
    "ROC-AUC change:",
    round(after_roc_auc - before_roc_auc, 4)
)

print(
    "Average Precision change:",
    round(after_ap - before_ap, 4)
)

print(
    "\nClient overlap must be zero:",
    len(client_overlap) == 0
)

Dataset shape: (30000, 44)
Number of features: 19

CLIENT GROUP CHECK
------------------
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0

BEFORE vs AFTER
----------------


,Evaluation,ROC-AUC,Average Precision
0,Week-5 random split,0.6777,0.6945
1,ML-09 client-grouped split,0.5950,0.5939


Metric change under grouped validation:
ROC-AUC change: -0.0826
Average Precision change: -0.1007

Client overlap must be zero: True


In [3]:
# ============================================================
# 12. ERROR EXAMPLES ON THE HONEST GROUPED TEST SET
# ============================================================

error_df = df.iloc[test_idx].copy()

error_df["actual"] = y_test_group.to_numpy()
error_df["score"] = group_scores

error_df["prediction"] = (
    error_df["score"] >= 0.5
).astype(int)

error_df["error_type"] = "Correct"

error_df.loc[
    (error_df["actual"] == 0) &
    (error_df["prediction"] == 1),
    "error_type"
] = "False Positive"

error_df.loc[
    (error_df["actual"] == 1) &
    (error_df["prediction"] == 0),
    "error_type"
] = "False Negative"


print("\nERROR SUMMARY")
print("-------------")

print(
    error_df["error_type"].value_counts()
)


# ------------------------------------------------------------
# False positives
# ------------------------------------------------------------

false_positives = error_df[
    error_df["error_type"] == "False Positive"
].copy()

print(
    "\nFalse positives:",
    len(false_positives)
)

display(
    false_positives[
        [
            "actual",
            "score",
            "prediction",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
            "word_count"
        ]
    ].head(5)
)


# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

false_negatives = error_df[
    error_df["error_type"] == "False Negative"
].copy()

print(
    "\nFalse negatives:",
    len(false_negatives)
)

display(
    false_negatives[
        [
            "actual",
            "score",
            "prediction",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
            "word_count"
        ]
    ].head(5)
)


ERROR SUMMARY
-------------
error_type
Correct           3534
False Positive    1495
False Negative    1134
Name: count, dtype: int64

False positives: 1495


,actual,score,prediction,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count
13,0,0.657224,1,307,0,0.00,39.8,238,103,1342.0
26,0,0.655660,1,2426,3,0.12,30.0,300,13,2686.0
36,0,0.725146,1,371,5,1.35,5.4,187,20,2510.0
64,0,0.775719,1,2639,3,0.11,7.2,106,8,2808.0
82,0,0.653449,1,1810,8,0.44,8.3,348,13,2793.0



False negatives: 1134


,actual,score,prediction,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count
23,1,0.279479,0,297,1,0.34,13.9,502,20,NaN
25,1,0.476854,0,27,0,0.00,7.2,180,20,2777.0
39,1,0.308167,0,4,0,0.00,36.3,348,104,3666.0
43,1,0.405949,0,184,0,0.00,2.9,138,20,2394.0
44,1,0.464176,0,64,0,0.00,55.8,90,20,3842.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I checked the final 19-feature set for direct label-derived and decision-derived fields.

The target is constructed from `trend_direction`, so `trend_direction` and `trend_pct` must not be model features.

I also checked the final feature list for identifiers and baseline decision scores that could create leakage.

The feature names do not contain the target field or the explicitly excluded trend fields.

A feature-name check cannot prove that all temporal leakage is absent. The starter content-level table does not expose a complete observation-date field, so I cannot establish a fully chronological feature-to-label separation from this table alone.

Therefore, the audit finds no obvious direct label leakage in the final feature list, while the temporal limitation remains documented.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

# Fields that should never be model features
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "target",
    "is_declining_label",
    "baseline_refresh_score"
]

print("FINAL MODEL FEATURES")
print("--------------------")

for feature in feature_cols:
    print(feature)

print("\nDIRECT LEAKAGE CHECK")
print("--------------------")

found_forbidden = [
    feature
    for feature in forbidden_features
    if feature in feature_cols
]

print(
    "Forbidden fields found in feature set:",
    found_forbidden
)

print(
    "No listed direct leakage fields:",
    len(found_forbidden) == 0
)


# ------------------------------------------------------------
# Check identifiers are not model features
# ------------------------------------------------------------

identifier_columns = [
    "content_id",
    "client_id"
]

identifier_leakage = [
    col
    for col in identifier_columns
    if col in feature_cols
]

print("\nIDENTIFIER CHECK")
print("----------------")

print(
    "Identifiers used as model features:",
    identifier_leakage
)

print(
    "No identifiers used as model features:",
    len(identifier_leakage) == 0
)


# ------------------------------------------------------------
# Check for suspicious target-like names
# ------------------------------------------------------------

target_words = [
    "target",
    "label",
    "trend_direction",
    "trend_pct",
    "outcome",
    "future",
    "next"
]

suspicious = [
    feature
    for feature in feature_cols
    if any(
        word in feature.lower()
        for word in target_words
    )
]

print("\nTARGET-LIKE FEATURE NAME CHECK")
print("-------------------------------")

print(
    "Suspicious feature names:",
    suspicious
)


# ------------------------------------------------------------
# Basic missing-value check
# ------------------------------------------------------------

print("\nMISSING VALUE CHECK")
print("-------------------")

print(
    "Total missing values:",
    X.isna().sum().sum()
)


# ------------------------------------------------------------
# Duplicate-row check
# ------------------------------------------------------------

print("\nDUPLICATE CHECK")
print("---------------")

duplicate_rows = X.duplicated().sum()

print(
    "Duplicate feature rows:",
    duplicate_rows
)


# ------------------------------------------------------------
# Client split check
# ------------------------------------------------------------

print("\nCLIENT SPLIT CHECK")
print("------------------")

print(
    "Training clients:",
    groups_train.nunique()
)

print(
    "Testing clients:",
    groups_test.nunique()
)

print(
    "Client overlap:",
    len(client_overlap)
)


# ------------------------------------------------------------
# Feature-target correlation screening
# ------------------------------------------------------------

correlations = (
    pd.concat(
        [
            X.reset_index(drop=True),
            y.reset_index(drop=True).rename("target")
        ],
        axis=1
    )
    .corr(numeric_only=True)["target"]
    .drop("target")
    .abs()
    .sort_values(ascending=False)
)

print("\nTOP ABSOLUTE FEATURE-TARGET CORRELATIONS")
print("-----------------------------------------")

display(
    correlations.head(10).to_frame(
        "absolute_correlation"
    )
)

print(
    "\nNote: correlation is only a screening signal; "
    "it does not by itself prove leakage."
)

FINAL MODEL FEATURES
--------------------
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
content_age_days
days_since_last_update
word_count
char_count
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct

DIRECT LEAKAGE CHECK
--------------------
Forbidden fields found in feature set: []
No listed direct leakage fields: True

IDENTIFIER CHECK
----------------
Identifiers used as model features: []
No identifiers used as model features: True

TARGET-LIKE FEATURE NAME CHECK
-------------------------------
Suspicious feature names: []

MISSING VALUE CHECK
-------------------
Total missing values: 0

DUPLICATE CHECK
---------------
Duplicate feature rows: 2

CLIENT SPLIT CHECK
------------------
Training clients: 25
Testing clients: 7
Client overlap: 0

TOP ABSOLUTE FEATURE-TARGET CORRELATIONS
-----------------------------------------


,absolute_correlation
days_with_impressions,0.190055
content_age_days,0.163882
word_count,0.118863
char_count,0.108025
days_since_last_update,0.081383
ctr,0.061911
clicks_90d,0.039680
engaged_sessions_90d,0.035402
avg_position,0.029035
days_with_sessions,0.025055



Note: correlation is only a screening signal; it does not by itself prove leakage.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

#### Earlier claim

The Logistic Regression model outperforms the Week-4 rule-based baseline and can identify content likely to decline.

#### Safer claim

On the Week-5 random holdout, the Logistic Regression model measured higher ROC-AUC and Average Precision than the rule-based baseline.

I then re-evaluated the same model under a client-grouped split, where entire clients were held out from training. This provides a stricter check of generalization across clients.

The results provide measured, directional evidence that the model may support prioritization of content for review. They do not establish causal effects, future production performance, or that the model will generalize to every new client.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 4 — CLAIM CHECK
# ============================================================

claim_summary = pd.DataFrame({
    "Evaluation": [
        "Week-5 random split",
        "ML-09 client-grouped split"
    ],
    "ROC-AUC": [
        before_roc_auc,
        after_roc_auc
    ],
    "Average Precision": [
        before_ap,
        after_ap
    ]
})

display(
    claim_summary.round(4)
)

print(
    "Claim language: observed / measured / directional / decision-support"
)

print(
    "Causal claim avoided: True"
)

print(
    "Production-performance claim avoided: True"
)

,Evaluation,ROC-AUC,Average Precision
0,Week-5 random split,0.6777,0.6945
1,ML-09 client-grouped split,0.5950,0.5939


Claim language: observed / measured / directional / decision-support
Causal claim avoided: True
Production-performance claim avoided: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.